In [1]:
include("SystemResponses.jl")
include("SymolicDerivative.jl")

using DifferentialEquations, SymbolicRegression, Logging, Interpolations
using .SymbolicDerivativeModule

binary_operators=(+, *, -, /)
unary_operators=(cos, sin, exp)

options = SymbolicRegression.Options(;
    binary_operators=binary_operators, 
    unary_operators=unary_operators,
    seed=42,
    # output_directory="output/deriv_search" # Save results to this file
)

# Define operators
operators = OperatorEnum(
    binary_operators=binary_operators,
    unary_operators=unary_operators
)

variable_names = ["x1", "x2", "x3"]

# Define variables (feature 2 is x2, feature 3 is x3)
x2 = Expression(Node{Float64}(feature=2); operators, variable_names)
x3 = Expression(Node{Float64}(feature=3); operators, variable_names)

# Construct the expression: -2 * x2 + 0.5 * x3
expression = -2.0 * x2 + 0.5 * x3
result = (x2 / -0.2419) + x3
wrong_function = 3.0 * x2 - x3 + 5.0

# Test evaluation
X = randn(Float64, 3, 100)
output = expression(X)

println("Expression: ", expression)
println("Result: ", result)

Expression: (-2.0 * x2) + (0.5 * x3)
Result: (x2 / -0.2419) + x3


In [5]:
tree_node = expression.tree

# Create the population member
member = PopMember(
    dataset,
    expression,
    options,
    deterministic=true
)
result_member = PopMember(
    dataset,
    result,
    options,
    deterministic=true
)


PopMember(tree = ((x2 / -0.2419) + x3), loss = 0.13467851091303237, cost = 0.13467851091303237)

In [6]:
function l2_loss_ocp(tree, dataset, options)
    # Extract time points, states, and control inputs from dataset
    # Assuming dataset.X has rows: [t; x; u]
    t = dataset.X[1, :]  # time points
    x_actual = dataset.y  # actual state trajectory
    u = dataset.X[3, :]  # control input
    
    # Initial condition
    x0 = x_actual[1]
    tspan = (t[1], t[end])
    
    # Create interpolation for control input
    u_interp = LinearInterpolation(t, u)
    
    # Define ODE function: dx/dt = tree([t, x, u])
    function ocp_dynamics(x, p, t_curr)
        # Check for invalid state
        if !isfinite(x) || !isfinite(t_curr)
            return Inf
        end
        
        # Get control input at current time
        u_curr = u_interp(t_curr)
        
        # Create input vector [t, x, u] for the tree (matching 3 features)
        input = reshape([t_curr, x, u_curr], :, 1)
        
        # Evaluate tree to get dx/dt
        try
            dx = tree(input)[1]
            # Check for valid output
            if !isfinite(dx)
                return Inf
            end
            return dx
        catch
            return Inf
        end
    end
    
    # Solve the ODE
    local loss  # Declare loss in function scope
    try
        ode_problem = ODEProblem(ocp_dynamics, x0, tspan)
        solution = solve(
            ode_problem,
            AutoTsit5(Rosenbrock23()),
            maxiters=5000,
            saveat=t,
            abstol=1e-3,
            reltol=1e-3,
            verbose=0
        )
        
        # Calculate loss
        if SciMLBase.successful_retcode(solution) && length(solution.u) == length(x_actual)
            x_predicted = [solution.u[i] for i in 1:length(solution.u)]
            # Check all predictions are finite
            if all(isfinite, x_predicted)
                loss = sum((x_predicted .- x_actual).^2) / length(x_actual)
            else
                loss = Inf
            end
        else
            loss = Inf
        end
    catch e
        # Any error during ODE solving results in infinite loss
        loss = Inf
    end
    
    return loss
end

l2_loss_ocp (generic function with 1 method)

In [7]:
expression_loss = l2_loss_ocp(expression.tree, dataset, options)
result_loss = l2_loss_ocp(result_member.tree, dataset, options)
wrong_loss = l2_loss_ocp(wrong_function.tree, dataset, options)

println("Expression Loss: ", expression_loss)
println("Result Loss: ", result_loss)
println("Wrong Function Loss: ", wrong_loss)

Expression Loss: 1.2848138010667562e-5
Result Loss: 0.0013543678027880563
Wrong Function Loss: 6.120702042492008e24


In [8]:
function symolic_integration(t_x_u, x, dominating_derivatives, options)
    guess_trees = [entry.tree for entry in dominating_derivatives]

    with_logger(NullLogger()) do
        println("--- Starting Search Using Integration Loss ---")
        hall_of_fame_ode = equation_search(
            t_x_u, x; 
            options=options,
            niterations = 5, # WARNING: This is very slow. 5 is just for a quick test.
                            # A real search might need 50+.
            parallelism = :multithreading,
            guesses = guess_trees
        )
        dominating_ode = calculate_pareto_frontier(hall_of_fame_ode)
    return dominating_ode
    end
end


symolic_integration (generic function with 1 method)

In [9]:
function l2_loss_multi_state(trees::Vector, t, x_matrix)
    """
    Loss function for multi-state ODE systems
    trees[i] computes dxi/dt
    t: time vector
    x_matrix: state matrix (n_time × n_states)
    """
    
    n_states = length(trees)
    n_time = length(t)
    
    # Initial conditions (first time point of each state)
    x0 = x_matrix[1, :]
    tspan = (t[1], t[end])
    
    # Define coupled ODE dynamics
    function multi_state_dynamics(x_vec, p, t_curr)
        # Check for invalid states
        if !all(isfinite, x_vec) || !isfinite(t_curr)
            return fill(Inf, n_states)
        end
        
        # Create input vector [t, x1, x2, ...]
        input = reshape(vcat([t_curr], x_vec), :, 1)
        
        # Evaluate each tree to get derivatives
        try
            dx = [trees[i](input)[1] for i in 1:n_states]
            
            # Check for valid outputs
            if !all(isfinite, dx)
                return fill(Inf, n_states)
            end
            
            return dx
        catch
            return fill(Inf, n_states)
        end
    end
    
    # Solve the coupled ODE system
    local loss
    try
        ode_problem = ODEProblem(multi_state_dynamics, x0, tspan)
        solution = solve(
            ode_problem,
            AutoTsit5(Rosenbrock23()),
            maxiters=5000,
            saveat=t,
            abstol=1e-3,
            reltol=1e-3,
            verbose=0
        )
        
        # Calculate loss
        if SciMLBase.successful_retcode(solution) && length(solution.u) == n_time
            # Convert solution to matrix (n_time × n_states)
            x_predicted = hcat([solution.u[i] for i in 1:length(solution.u)]...)'
            
            # Check all predictions are finite
            if all(isfinite, x_predicted)
                # Mean squared error across all states
                loss = sum((x_predicted .- x_matrix).^2) / length(x_matrix)
            else
                loss = Inf
            end
        else
            loss = Inf
        end
    catch e
        loss = Inf
    end
    
    return loss
end

function symbolic_integration_multi_state(t, x_matrix, dominating_derivatives_multi, options)
    """
    Refine multi-state derivative equations using integration-based loss
    Tests ALL combinations of candidates up to complexity 10
    
    Parameters:
    - t: time vector
    - x_matrix: state matrix (n_time × n_states)
    - dominating_derivatives_multi: Vector of results, one per state
    - options: SymbolicRegression options
    """
    
    n_states = length(dominating_derivatives_multi)
    
    println("=== Multi-State Integration-Based Refinement ===")
    println("Testing ALL combinations of candidates (complexity ≤ 10)...\n")
    
    # Get ALL candidates for each state (no limit, since complexity is already limited to 10)
    candidates_per_state = [length(results) for results in dominating_derivatives_multi]
    
    println("Number of candidates per state: ", candidates_per_state)
    println("Total combinations to test: ", prod(candidates_per_state))
    println()
    
    # Store best combination
    best_loss = Inf
    best_trees = nothing
    best_indices = nothing
    
    # Counter for progress
    combinations_tested = 0
    total_combinations = prod(candidates_per_state)
    
    # Try all combinations
    function try_combinations(state_idx, current_trees, current_indices)
        if state_idx > n_states
            # Evaluate this combination
            combinations_tested += 1
            
            # Progress indicator
            if combinations_tested % max(1, div(total_combinations, 10)) == 0
                progress_pct = round(100 * combinations_tested / total_combinations, digits=1)
                println("Progress: $combinations_tested/$total_combinations ($progress_pct%)")
            end
            
            loss = l2_loss_multi_state(current_trees, t, x_matrix)
            
            if loss < best_loss
                best_loss = loss
                best_trees = copy(current_trees)
                best_indices = copy(current_indices)
            end
            return
        end
        
        # Try each candidate for current state
        n_candidates = candidates_per_state[state_idx]
        for i in 1:n_candidates
            tree = dominating_derivatives_multi[state_idx][i].tree
            push!(current_trees, tree)
            push!(current_indices, i)
            
            try_combinations(state_idx + 1, current_trees, current_indices)
            
            pop!(current_trees)
            pop!(current_indices)
        end
    end
    
    # Start recursive search
    try_combinations(1, [], [])
    
    # Display results
    println("\n" * "="^60)
    println("Best combination found:")
    println("="^60)
    for (i, (tree, idx)) in enumerate(zip(best_trees, best_indices))
        member = dominating_derivatives_multi[i][idx]
        complexity = compute_complexity(member, options)
        println("\nState $i (candidate #$idx out of $(candidates_per_state[i])):")
        println("  Derivative loss: ", round(member.loss, sigdigits=4))
        println("  Complexity: ", complexity)
        println("  Expression: dx$i/dt = ", string_tree(tree, options))
    end
    
    println("\n" * "="^60)
    println("Integration loss: ", round(best_loss, sigdigits=4))
    println("="^60)
    
    return best_trees, best_loss, best_indices
end

symbolic_integration_multi_state (generic function with 1 method)

In [10]:
# Test the multi-state integration
refined_trees, integration_loss, best_indices = symbolic_integration_multi_state(
    t_pp, 
    x_matrix, 
    dominating_derivatives_multi, 
    options
)

UndefVarError: UndefVarError: `x_matrix` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [11]:
# Generate data for predator-prey system
t_pp = 0:0.1:50
x1_pp, x2_pp = SystemResponsesModule.predator_prey_system(t_pp; alpha=1.0, beta=0.1, delta=0.075, gamma=1.5, x1_0=10.0, x2_0=5.0, error_std=0.1)

# Create state matrix
x_matrix = hcat(x1_pp, x2_pp)  # Shape: (n_time, n_states)

# Use multi-state symbolic derivative
dominating_derivatives_multi = SymbolicDerivativeModule.symbolic_derivative_multi_state(t_pp, x_matrix)

Searching for derivative of state 1...


nfo: Started!












Evolving for 10 iterations... 100%|██████████████████████| Time: 0:00:08


───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           1.514e+03  0.000e+00  y = -0.1961
2           1.501e+03  8.649e-03  y = cos(x₁)
3           7.870e+02  6.456e-01  y = 7.2664 - x₃
5           7.451e+02  2.736e-02  y = (x₃ / -0.76653) + 9.5394
6           4.891e+02  4.209e-01  y = (x₂ / exp(x₃)) - x₃
8           4.831e+02  6.268e-03  y = (x₂ / exp(x₃)) + (2.4685 - x₃)
10          4.793e+02  3.857e-03  y = (4.9137 - x₃) + ((x₂ - 4.2194) / exp(x₃))
───────────────────────────────────────────────────────────────────────────────────────────────────


┌ Info: Final population:
└ @ SymbolicRegression /home/fidelius/SymbolicRegression.jl/src/SymbolicRegression.jl:1223


  - outputs/20251218_145011_tax4bI/hall_of_fame.csv
Searching for derivative of state 2...

┌ Info: Results saved to:
└ @ SymbolicRegression /home/fidelius/SymbolicRegression.jl/src/SymbolicRegression.jl:1246


nfo: Started!









































Evolving for 10 iterations... 100%|██████████████████████| Time: 0:00:02


───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           8.938e+02  0.000e+00  y = -0.10147
2           8.849e+02  9.995e-03  y = sin(x₁)
3           6.073e+02  3.764e-01  y = x₂ * 0.48566
4           5.459e+02  1.067e-01  y = exp(x₂ * 0.023955)
5           5.367e+02  1.686e-02  y = x₂ * (x₂ * 0.0034859)
6           5.329e+02  7.201e-03  y = exp(x₂ * 0.023955) + -2.6999
7           4.915e+02  8.077e-02  y = (x₃ + x₂) * (x₂ * 0.0033944)
9           4.389e+02  5.665e-02  y = x₂ / exp(2.3544 / exp(x₃ / 3.6096))
───────────────────────────────────────────────────────────────────────────────────────────────────
  - outputs/20251218_145030_T65Ve0/hall_of_fame.csv


┌ Info: Final population:
└ @ SymbolicRegression /home/fidelius/SymbolicRegression.jl/src/SymbolicRegression.jl:1223
┌ Info: Results saved to:
└ @ SymbolicRegression /home/fidelius/SymbolicRegression.jl/src/SymbolicRegression.jl:1246


2-element Vector{Any}:
 PopMember{Float64, Float64, Expression{Float64, Node{Float64, 2}, @NamedTuple{operators::OperatorEnum{Tuple{Tuple{typeof(cos), typeof(sin), typeof(exp)}, Tuple{typeof(+), typeof(*), typeof(/), typeof(-)}}}, variable_names::Vector{String}}}}[PopMember(tree = (-0.1961026673018221), loss = 1513.982914111372, cost = 0.9999745999204945), PopMember(tree = (cos(x1)), loss = 1500.9443015340864, cost = 0.9913626920356936), PopMember(tree = (7.266421734880739 - x3), loss = 787.0200013083622, cost = 0.519820932985683), PopMember(tree = ((x3 / -0.7665286074631106) + 9.539377946978732), loss = 745.1095677505151, cost = 0.4921393993046366), PopMember(tree = ((x2 / exp(x3)) - x3), loss = 489.1499148624819, cost = 0.3230799276904998), PopMember(tree = ((x2 / exp(x3)) + (2.468543242331126 - x3)), loss = 483.05620977320166, cost = 0.31905508021574275), PopMember(tree = ((4.913706697711076 - x3) + ((x2 - 4.2194059012998215) / exp(x3))), loss = 479.3444612323966, cost = 0.316603497

In [12]:
# Display results for each state
for (state_idx, results) in enumerate(dominating_derivatives_multi)
    println("\n" * "="^60)
    println("State $state_idx Derivative Equations (dx$state_idx/dt):")
    println("="^60)
    
    for (i, member) in enumerate(results)
        println("Exp: ", string_tree(member.tree, options))
    end
end


State 1 Derivative Equations (dx1/dt):
Exp: -0.1961026673018221
Exp: cos(x1)
Exp: 7.266421734880739 / x3
Exp: (x3 - -0.7665286074631106) + 9.539377946978732
Exp: (x2 - exp(x3)) / x3
Exp: (x2 - exp(x3)) + (2.468543242331126 / x3)
Exp: (4.913706697711076 / x3) + ((x2 / 4.2194059012998215) - exp(x3))

State 2 Derivative Equations (dx2/dt):
Exp: -0.10146523060875003
Exp: sin(x1)
Exp: x2 * 0.48566421874958443
Exp: exp(x2 * 0.023955263860955724)
Exp: x2 * (x2 * 0.0034859196736582256)
Exp: exp(x2 * 0.023955263640788222) + -2.699942381754305
Exp: (x3 + x2) * (x2 * 0.0033943641815677435)
Exp: x2 - exp(2.354353016732027 - exp(x3 - 3.609574905009916))


In [13]:
integration_options = SymbolicRegression.Options(;
    binary_operators=binary_operators, 
    unary_operators=unary_operators,
    maxsize=10,  # Limit complexity to match derivative search
    seed=42,
    loss_function=l2_loss_ocp
    # output_directory="output/deriv_search" # Save results to this file
)

symolic_integration(t_x_u, x, dominating_derivatives, integration_options)

UndefVarError: UndefVarError: `dominating_derivatives` not defined in `Main`
Suggestion: check for spelling errors or missing imports.